In [1]:
import pandas as pd
import os
import csv
from scipy.special import softmax
import seaborn as sns
import matplotlib.pyplot as plt
import re
import math

In [19]:
def softmax_with_alpha(values, alpha=10.0):
    """
    Compute the softmax of a list of values with an alpha scaling parameter.

    Args:
        values (list): A list of numeric values.
        alpha (float): Scaling parameter for sharpness/smoothness (default = 1.0).

    Returns:
        list: Softmax probabilities.
    """
    # Subtract the max value for numerical stability
    # max_val = max(values)
    exp_values = [math.exp(alpha * v ) for v in values]
    total = sum(exp_values)
    return [v / total for v in exp_values]

In [28]:
df = pd.read_csv('gemini-1.5-pro_speaker_test_1_1_0_iter0_predicted_answers_scoring_fixedPrompt.csv')
df

,halo,goal,affect_valence,name,utterance,state,item,parsed_answer
0,fuzzy,state,0,Tom,$50,$50,electric kettle,A:1.0
1,fuzzy,state,0,Tom,$51,$50,electric kettle,A:0.9
2,fuzzy,state,0,Tom,$500,$50,electric kettle,A:0.01
3,fuzzy,state,0,Tom,$501,$50,electric kettle,A:0.0
4,fuzzy,state,0,Tom,$1000,$50,electric kettle,A:0.0
...,...,...,...,...,...,...,...,...
3595,exact,both,1,Robert,$1001,$10001,watch,A:0.0
3596,exact,both,1,Robert,$5000,$10001,watch,A:0
3597,exact,both,1,Robert,$5001,$10001,watch,A:0
3598,exact,both,1,Robert,$10000,$10001,watch,A:0.0


In [29]:
df["item"].unique()

array(['electric kettle', 'laptop', 'watch'], dtype=object)

In [30]:
# filter out responses which do not contain floats or are empty
def clean_llm_predictions(r, is_list = False):
    r = str(r)
    float_regex = r"-?\d*\.\d+|\d+"
    # check that there are float regex matches in r
    match = re.findall(float_regex, r)
    # check if matches are empty and a prediction was made, and the runtime processing didn't return 1000 which was used for erroneous strings
    if (len(match) > 0): # and (match[0] != "1000")
        if not is_list:
            res = match[0]
        else:
            res = match
    else:
        res = None
    return float(res) if res is not None else None

In [31]:
df["utterance"] = df["utterance"].apply(lambda x: int(x.replace("$", "")))
df["state"] = df["state"].apply(lambda x: int(x.replace("$", "")))
# df["parsed_answer"] = df["parsed_answer"].apply(lambda x: float(x.replace("A:", "").replace("<", "").replace(">", "").strip()))
df["parsed_answer"] = df["parsed_answer"].apply(lambda r: clean_llm_predictions(r))
df

,halo,goal,affect_valence,name,utterance,state,item,parsed_answer
0,fuzzy,state,0,Tom,50,50,electric kettle,1.00
1,fuzzy,state,0,Tom,51,50,electric kettle,0.90
2,fuzzy,state,0,Tom,500,50,electric kettle,0.01
3,fuzzy,state,0,Tom,501,50,electric kettle,0.00
4,fuzzy,state,0,Tom,1000,50,electric kettle,0.00
...,...,...,...,...,...,...,...,...
3595,exact,both,1,Robert,1001,10001,watch,0.00
3596,exact,both,1,Robert,5000,10001,watch,0.00
3597,exact,both,1,Robert,5001,10001,watch,0.00
3598,exact,both,1,Robert,10000,10001,watch,0.00


In [33]:
# renormalize after summing over the cells within a partition for the relevant goals
df_utterance_probs_affect = df[df['goal'] == 'affect']
# sum over the probs of each utterance across states
df_utterance_probs_affect_summed = df_utterance_probs_affect.groupby(['item', 'affect_valence', 'utterance']).apply(lambda x: sum(x['parsed_answer'])).reset_index()
df_utterance_probs_affect_summed = df_utterance_probs_affect_summed.rename(columns={0: 'utterance_prob_summed'})
# nore renormalize
df_utterance_probs_affect_renorm = (
    df_utterance_probs_affect_summed
    .groupby(['item', 'affect_valence'])
    .apply(lambda x: pd.DataFrame({
        'utterance': x['utterance'].tolist(),
        'utterance_prob': softmax_with_alpha(x['utterance_prob_summed'], alpha=1)
    }))
    .reset_index(level=[0, 1])  # Reset only the grouping indices
)


/var/folders/fn/6ct_6l112376k8288ws7798m0000gn/T/ipykernel_58220/3325983077.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_utterance_probs_affect_summed = df_utterance_probs_affect.groupby(['item', 'affect_valence', 'utterance']).apply(lambda x: sum(x['parsed_answer'])).reset_index()
/var/folders/fn/6ct_6l112376k8288ws7798m0000gn/T/ipykernel_58220/3325983077.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this

In [34]:
# for state only goal, we sum over the different affect values
df_utterance_probs_state_exact = df[(df['goal'] == 'state') & (df["halo"] == 'exact')]
df_utterance_probs_state_summed = df_utterance_probs_state_exact.groupby(['item', 'state', 'utterance']).apply(lambda x: sum(x['parsed_answer'])).reset_index()
df_utterance_probs_state_summed = df_utterance_probs_state_summed.rename(columns={0: 'utterance_prob_summed'})
# now renormalize
df_utterance_probs_state_renorm = (
    df_utterance_probs_state_summed
    .groupby(['item', 'state'])
    .apply(lambda x: pd.DataFrame({
        'utterance': x['utterance'].tolist(),
        'utterance_prob': softmax_with_alpha(x['utterance_prob_summed'], alpha=1)
    }))
    .reset_index(level=[0, 1])  # Reset only the grouping indices
)
df_utterance_probs_state_renorm

/var/folders/fn/6ct_6l112376k8288ws7798m0000gn/T/ipykernel_58220/607799947.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_utterance_probs_state_summed = df_utterance_probs_state_exact.groupby(['item', 'state', 'utterance']).apply(lambda x: sum(x['parsed_answer'])).reset_index()
/var/folders/fn/6ct_6l112376k8288ws7798m0000gn/T/ipykernel_58220/607799947.py:9: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning

,item,state,utterance,utterance_prob
0,electric kettle,50,50,0.450853
1,electric kettle,50,51,0.061016
2,electric kettle,50,500,0.061016
3,electric kettle,50,501,0.061016
4,electric kettle,50,1000,0.061016
...,...,...,...,...
5,watch,10001,1001,0.061016
6,watch,10001,5000,0.061016
7,watch,10001,5001,0.061016
8,watch,10001,10000,0.061016


In [35]:
# for fuzzy states, we round the states and collapse across them
df_utterance_probs_state_fuzzy = df[(df['goal'] == 'state') & (df["halo"] == 'fuzzy')]
df_utterance_probs_state_fuzzy['state'] = df_utterance_probs_state_fuzzy['state'].apply(lambda x: round(x, -1))
df_utterance_probs_state_fuzzy_summed = df_utterance_probs_state_fuzzy.groupby(['item', 'state', 'utterance']).apply(lambda x: sum(x['parsed_answer'])).reset_index()
df_utterance_probs_state_fuzzy_summed = df_utterance_probs_state_fuzzy_summed.rename(columns={0: 'utterance_prob_summed'})
# now renormalize
df_utterance_probs_state_fuzzy_renorm = (
    df_utterance_probs_state_fuzzy_summed
    .groupby(['item', 'state'])
    .apply(lambda x: pd.DataFrame({
        'utterance': x['utterance'].tolist(),
        'utterance_prob': softmax_with_alpha(x['utterance_prob_summed'], alpha=1)
    }))
    .reset_index(level=[0, 1])  # Reset only the grouping indices
)
df_utterance_probs_state_fuzzy_renorm

/var/folders/fn/6ct_6l112376k8288ws7798m0000gn/T/ipykernel_58220/2953501181.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_utterance_probs_state_fuzzy['state'] = df_utterance_probs_state_fuzzy['state'].apply(lambda x: round(x, -1))
/var/folders/fn/6ct_6l112376k8288ws7798m0000gn/T/ipykernel_58220/2953501181.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_utterance_probs_state_fuzzy_summed = df_utterance_probs_state_fuzzy.groupby(['item', 'stat

,item,state,utterance,utterance_prob
0,electric kettle,50,50,0.458646
1,electric kettle,50,51,0.458646
2,electric kettle,50,500,0.010679
3,electric kettle,50,501,0.010468
4,electric kettle,50,1000,0.010260
...,...,...,...,...
5,watch,10000,1001,0.014525
6,watch,10000,5000,0.014819
7,watch,10000,5001,0.014525
8,watch,10000,10000,0.793058


In [36]:
# now we do the remainin goal of both s, a
# fuzzy first
df_utterance_probs_both_fuzzy = df[(df['goal'] == 'both') & (df["halo"] == 'fuzzy')]
df_utterance_probs_both_fuzzy['state'] = df_utterance_probs_both_fuzzy['state'].apply(lambda x: round(x, -1))
df_utterance_probs_both_fuzzy_summed = df_utterance_probs_both_fuzzy.groupby(['item', 'state', 'affect_valence', 'utterance']).apply(lambda x: sum(x['parsed_answer'])).reset_index()
df_utterance_probs_both_fuzzy_summed = df_utterance_probs_both_fuzzy_summed.rename(columns={0: 'utterance_prob_summed'})
df_utterance_probs_both_fuzzy_summed
# now renormalize
df_utterance_probs_both_fuzzy_renorm = (
    df_utterance_probs_both_fuzzy_summed
    .groupby(['item', 'state', 'affect_valence'])
    .apply(lambda x: pd.DataFrame({
        'utterance': x['utterance'].tolist(),
        'utterance_prob': softmax_with_alpha(x['utterance_prob_summed'], alpha=1)
    }))
    .reset_index(level=[0, 1, 2])  # Reset only the grouping indices
)
df_utterance_probs_both_fuzzy_renorm

/var/folders/fn/6ct_6l112376k8288ws7798m0000gn/T/ipykernel_58220/729227339.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_utterance_probs_both_fuzzy['state'] = df_utterance_probs_both_fuzzy['state'].apply(lambda x: round(x, -1))
/var/folders/fn/6ct_6l112376k8288ws7798m0000gn/T/ipykernel_58220/729227339.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_utterance_probs_both_fuzzy_summed = df_utterance_probs_both_fuzzy.groupby(['item', 'state', 'a

,item,state,affect_valence,utterance,utterance_prob
0,electric kettle,50,0,50,0.312395
1,electric kettle,50,0,51,0.312395
2,electric kettle,50,0,500,0.047668
3,electric kettle,50,0,501,0.047194
4,electric kettle,50,0,1000,0.046725
...,...,...,...,...,...
5,watch,10000,1,1001,0.058842
6,watch,10000,1,5000,0.058256
7,watch,10000,1,5001,0.058256
8,watch,10000,1,10000,0.389494


In [37]:
# exact
df_utterance_probs_both_exact = df[(df['goal'] == 'both') & (df["halo"] == 'exact')]
df_utterance_probs_both_exact_summed = df_utterance_probs_both_exact.groupby(['item', 'state', 'affect_valence', 'utterance']).apply(lambda x: sum(x['parsed_answer'])).reset_index()
df_utterance_probs_both_exact_summed = df_utterance_probs_both_exact_summed.rename(columns={0: 'utterance_prob_summed'})
df_utterance_probs_both_exact_summed
# now renormalize
df_utterance_probs_both_exact_renorm = (
    df_utterance_probs_both_exact_summed
    .groupby(['item', 'state', 'affect_valence'])
    .apply(lambda x: pd.DataFrame({
        'utterance': x['utterance'].tolist(),
        'utterance_prob': softmax_with_alpha(x['utterance_prob_summed'], alpha=1)
    }))
    .reset_index(level=[0, 1, 2])  # Reset only the grouping indices
)
df_utterance_probs_both_exact_renorm

/var/folders/fn/6ct_6l112376k8288ws7798m0000gn/T/ipykernel_58220/2805667727.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_utterance_probs_both_exact_summed = df_utterance_probs_both_exact.groupby(['item', 'state', 'affect_valence', 'utterance']).apply(lambda x: sum(x['parsed_answer'])).reset_index()
/var/folders/fn/6ct_6l112376k8288ws7798m0000gn/T/ipykernel_58220/2805667727.py:10: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupb

,item,state,affect_valence,utterance,utterance_prob
0,electric kettle,50,0,50,0.231969
1,electric kettle,50,0,51,0.085337
2,electric kettle,50,0,500,0.085337
3,electric kettle,50,0,501,0.085337
4,electric kettle,50,0,1000,0.085337
...,...,...,...,...,...
5,watch,10001,1,1001,0.085337
6,watch,10001,1,5000,0.085337
7,watch,10001,1,5001,0.085337
8,watch,10001,1,10000,0.085337


In [38]:
df_utterance_probs_state_renorm

,item,state,utterance,utterance_prob
0,electric kettle,50,50,0.450853
1,electric kettle,50,51,0.061016
2,electric kettle,50,500,0.061016
3,electric kettle,50,501,0.061016
4,electric kettle,50,1000,0.061016
...,...,...,...,...
5,watch,10001,1001,0.061016
6,watch,10001,5000,0.061016
7,watch,10001,5001,0.061016
8,watch,10001,10000,0.061016


In [39]:
# combine all renormalized dataframes
df_utterance_probs_affect_renorm["halo"] = "exact"
df_utterance_probs_affect_renorm["goal"] = "affect"
df_utterance_probs_affect_renorm["state"] = 0
df_utterance_probs_state_renorm["halo"] = "exact"
df_utterance_probs_state_renorm["goal"] = "state"
df_utterance_probs_state_renorm["affect_valence"] = 0
df_utterance_probs_state_fuzzy_renorm["halo"] = "fuzzy"
df_utterance_probs_state_fuzzy_renorm["goal"] = "state"
df_utterance_probs_state_fuzzy_renorm["affect_valence"] = 0
df_utterance_probs_both_fuzzy_renorm["halo"] = "fuzzy"
df_utterance_probs_both_fuzzy_renorm["goal"] = "both"
df_utterance_probs_both_exact_renorm["halo"] = "exact"
df_utterance_probs_both_exact_renorm["goal"] = "both"

df_utterance_probs_renorm = pd.concat(
    [
        df_utterance_probs_affect_renorm, 
        df_utterance_probs_state_renorm, 
        df_utterance_probs_state_fuzzy_renorm, 
        df_utterance_probs_both_fuzzy_renorm, 
        df_utterance_probs_both_exact_renorm
    ]
)
df_utterance_probs_renorm

,item,affect_valence,utterance,utterance_prob,halo,goal,state
0,electric kettle,0,50,0.150811,exact,affect,0
1,electric kettle,0,51,0.031375,exact,affect,0
2,electric kettle,0,500,0.184201,exact,affect,0
3,electric kettle,0,501,0.031407,exact,affect,0
4,electric kettle,0,1000,0.203573,exact,affect,0
...,...,...,...,...,...,...,...
5,watch,1,1001,0.085337,exact,both,10001
6,watch,1,5000,0.085337,exact,both,10001
7,watch,1,5001,0.085337,exact,both,10001
8,watch,1,10000,0.085337,exact,both,10001


In [40]:
# df_utterance_probs_renorm.to_csv("../../data/results_pt_01_14/speaker_gpt4omini/gemini_renormalized_utterance_probs_fixedPrompt_alpha1.csv", index=False)

In [15]:
# categorize the predictions

def categorize_s_u_pair(row):
    state = row["state"]
    utterance = row["utterance"]
    if state == utterance:
        interpretation = "exact"
    elif abs(state - utterance) < 5:
        interpretation = "fuzzy"
    elif utterance > state:
        interpretation = "hyperbolic"
    else:
        interpretation = "other"

    return interpretation

In [104]:
# def classify_condition(row):
#     if (row["affect_valence"] == 1) and (row["affect"] == "a"):
#         condition = "hyperbole_1"
#     elif (row["affect_valence"] == 0) and (row["affect"] == "a"):
#         condition = "hyperbole_0"
#     elif 

In [16]:
df_utterance_probs_categorized = df_utterance_probs_renorm.copy()
df_utterance_probs_categorized["interpretation"] = df_utterance_probs_categorized.apply(categorize_s_u_pair, axis=1)

In [17]:
df_utterance_probs_categorized

,item,affect_valence,utterance,utterance_prob,halo,goal,state,interpretation
0,electric kettle,0,50,0.127652,exact,affect,0,hyperbolic
1,electric kettle,0,51,0.090859,exact,affect,0,hyperbolic
2,electric kettle,0,500,0.116665,exact,affect,0,hyperbolic
3,electric kettle,0,501,0.078203,exact,affect,0,hyperbolic
4,electric kettle,0,1000,0.155915,exact,affect,0,hyperbolic
...,...,...,...,...,...,...,...,...
5,watch,1,1001,0.091324,exact,both,10001,other
6,watch,1,5000,0.091324,exact,both,10001,other
7,watch,1,5001,0.082633,exact,both,10001,other
8,watch,1,10000,0.100928,exact,both,10001,fuzzy


In [6]:
## analyse free generation results
df = pd.read_csv("../../data/results_pt_01_14/gpt-4o-mini_speaker_test_1_1_0_iter0_predicted_answers_updatedFreeGeneration_noState.csv")
df.head()

,halo,goal,affect_valence,name,item,parsed_answer
0,fuzzy,state,0,Fred,electric kettle,A: 50
1,fuzzy,state,0,Fred,electric kettle,A: 30
2,fuzzy,state,0,Fred,electric kettle,A: 30
3,fuzzy,state,0,Fred,electric kettle,A: 50
4,fuzzy,state,0,Fred,electric kettle,A: 50


In [9]:
df["parsed_answer_clean"] = df["parsed_answer"].apply(lambda r: clean_llm_predictions(r))
df

,halo,goal,affect_valence,name,item,parsed_answer,parsed_answer_clean
0,fuzzy,state,0,Fred,electric kettle,A: 50,50.0
1,fuzzy,state,0,Fred,electric kettle,A: 30,30.0
2,fuzzy,state,0,Fred,electric kettle,A: 30,30.0
3,fuzzy,state,0,Fred,electric kettle,A: 50,50.0
4,fuzzy,state,0,Fred,electric kettle,A: 50,50.0
...,...,...,...,...,...,...,...
355,exact,both,1,Frank,watch,A: 500,500.0
356,exact,both,1,Frank,watch,A: 500,500.0
357,exact,both,1,Frank,watch,A: 500,500.0
358,exact,both,1,Frank,watch,A: 300,300.0


In [10]:
df = df.dropna(subset=["parsed_answer_clean"])
print(len(df))

360


In [11]:
# analyse trends by goal
df_affect = df[df["goal"]=="affect"]
df_affect

,halo,goal,affect_valence,name,item,parsed_answer,parsed_answer_clean
20,fuzzy,affect,0,Fred,electric kettle,A: 50,50.0
21,fuzzy,affect,0,Fred,electric kettle,A: 50,50.0
22,fuzzy,affect,0,Fred,electric kettle,A: 50,50.0
23,fuzzy,affect,0,Fred,electric kettle,A: 50,50.0
24,fuzzy,affect,0,Fred,electric kettle,A: 50,50.0
...,...,...,...,...,...,...,...
335,exact,affect,1,Frank,watch,A: 500,500.0
336,exact,affect,1,Frank,watch,A: 200,200.0
337,exact,affect,1,Frank,watch,A: 300,300.0
338,exact,affect,1,Frank,watch,A: 300,300.0


In [22]:
df[df["goal"]!="state"].groupby("affect_valence").mean('parsed_answer_clean')

,parsed_answer_clean
affect_valence,
0,342.333333
1,590.125000
